In [1]:
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
import re
from ast import literal_eval
import re

def clean_to_sent(text, min_line_len=40, min_words=5):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)
    text = text.strip()
    text = re.sub(r'Tweede Kamer.*?\d{4,}', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Vergaderjaar\s*\d{4}–\d{4}', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bNr\.\s*\d+\b', '', text)
    text = re.sub(r'ISSN\s*\d+', '', text)
    text = re.sub(r'\bkst-\d+-\d+\b', '', text)
    text = re.sub(r'\bPagina\s*\d+\b', '', text)
    text = re.sub(r'\bBladzijde\s*\d+\b', '', text)

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    merged = []
    for line in lines:
        if re.match(r'^(De voorzitter|Griffier:|Aanvang\s|Voorzitter:|Van dit overleg|De griffier)', line):
            continue
        if merged:
            prev = merged[-1]
            if not re.search(r'[.!?]"?$', prev) and re.match(r'^[a-zéèà]', line):
                merged[-1] = prev.rstrip('-') + ' ' + line
                continue
        merged.append(line)

    merged = [l for l in merged if len(l) >= min_line_len or re.search(r'[.!?]"?$', l)]
    text = ' '.join(merged)
    text = re.sub(r'\s+', ' ', text)

    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.split()) >= min_words and not re.match(r'^[A-Z\s]+$', s)]
    return sentences


In [ ]:
# Clean KST

kst = pd.read_excel('data/kst.xlsx',usecols=['_id','doc_content','meta','published_at'])
kst_meta = pd.DataFrame([{k:v for k,v in literal_eval(m).items() if type(v) == str} for i,m in zip(kst._id,kst.meta)])
kst = pd.concat([kst.drop(columns=['meta']).reset_index(drop=True),
                          kst_meta.reset_index(drop=True)], axis=1)
kst = kst.rename(columns={"doc_content":"text", "published_at":"date","_id":"id"})
kst = kst.dropna(subset=['text'])
kst['sentences'] = kst['text'].apply(lambda x: clean_to_sent(x, min_words=15))
kst = kst[kst['sentences'].apply(len) > 0].reset_index(drop=True)
kst = kst.drop(columns=['text']).explode('sentences').reset_index(drop=True)
kst['sent_id'] = kst.groupby('id').cumcount() + 1
kst['sid'] = kst['id'] + '_s' + kst['sent_id'].astype(str)
kst = kst.rename(columns={"sentences":"sentence"})
kst.to_csv('data/processed/sentences-kst.csv',sep='\t',index=False)

In [ ]:
# Clean Report

kst = pd.read_excel('data/rekenkamer.xlsx',usecols=['_id','doc_content','meta','published_at'])
kst_meta = pd.DataFrame([{k:v for k,v in literal_eval(m).items() if type(v) == str} for i,m in zip(kst._id,kst.meta)])
kst = pd.concat([kst.drop(columns=['meta']).reset_index(drop=True),
                          kst_meta.reset_index(drop=True)], axis=1)
kst = kst.rename(columns={"doc_content":"text", "published_at":"date","_id":"id"})
kst = kst.dropna(subset=['text'])
kst['sentences'] = kst['text'].apply(lambda x: clean_to_sent(x, min_words=15))
kst = kst[kst['sentences'].apply(len) > 0].reset_index(drop=True)
kst = kst.drop(columns=['text']).explode('sentences').reset_index(drop=True)
kst['sent_id'] = kst.groupby('id').cumcount() + 1
kst['sent_id'] = kst['id'] + '_s' + kst['sent_id'].astype(str)
kst = kst.rename(columns={"sentences":"sentence"})
kst.to_csv('data/processed/sentences-reports.csv',sep='\t',index=False)

In [7]:
# Clean TK

df = pd.read_csv('data/tk.csv',sep='\t',usecols=['p_id', 'text', 'member-ref', 'party-ref', 'role', 'date', 'house'])
df = df.dropna(subset=['text']).rename(columns={"p_id":"id"})
df['sentence'] = df['text'].apply(lambda x: clean_to_sent(x, min_words=15))
df = df[df['sentence'].apply(len) > 0].reset_index(drop=True)
df = df.drop(columns=['text']).explode('sentence').reset_index(drop=True)
df['sent_id'] = df.groupby('id').cumcount() + 1
df['sent_id'] = df['id'] + '_s' + df['sent_id'].astype(str)
df = df.rename(columns={"sentences":"sentence"})
df.to_csv('data/processed/sentences-tk.csv',sep='\t',index=False)

/tmp/ipykernel_9865/4079332487.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/tk.csv',sep='\t',usecols=['p_id', 'text', 'member-ref', 'party-ref', 'role', 'date', 'house'])


KeyboardInterrupt: 

In [6]:
df

,p_id,text,member-ref,party-ref,role,date,house
0,nl.proc.ob.d.h-tk-20032004-5053-5075.1.7.1.1,Voorzitter. De CDA-fractie is nooit echt warm ...,nl.m.02370,nl.p.cda,mp,2004-05-27,commons
1,nl.proc.ob.d.h-tk-20032004-5053-5075.1.7.1.2,"De CDA-fractie heeft altijd de afnemer, klein ...",nl.m.02370,nl.p.cda,mp,2004-05-27,commons
2,nl.proc.ob.d.h-tk-20032004-5053-5075.1.7.2.1,U heeft een amendement ingediend over de flexi...,nl.m.02414,nl.p.vvd,mp,2004-05-27,commons
3,nl.proc.ob.d.h-tk-20032004-5053-5075.1.7.3.1,Ik zal in het volgende deel van mijn betoog pr...,nl.m.02370,nl.p.cda,mp,2004-05-27,commons
4,nl.proc.ob.d.h-tk-20032004-5053-5075.1.7.3.2,Ik vraag aandacht voor een aantal zwakke groep...,nl.m.02370,nl.p.cda,mp,2004-05-27,commons
...,...,...,...,...,...,...,...
2350264,ParlaMint-NL_2022-03-30-tweedekamer-2.u97,Ze zaten niet in het rapport van de staatscomm...,NaN,CU,unknown,2022-03-30,commons
2350265,ParlaMint-NL_2022-03-30-tweedekamer-2.u98,Het is tweeledig. Aan de ene kant geldt op bas...,NaN,CDA,government,2022-03-30,commons
2350266,ParlaMint-NL_2022-03-30-tweedekamer-2.u99,Daar ga ik graag op in. De aanleiding hiervoor...,NaN,CU,unknown,2022-03-30,commons
2350267,ParlaMint-NL_2022-03-30-tweedekamer-2.u101,We vragen de regering op een laagdrempelige ma...,NaN,CU,unknown,2022-03-30,commons
